# PatrolIQ — MLflow Experiment Tracking

This notebook records the selected PatrolIQ clustering and dimensionality-reduction workflow with MLflow.

It tracks geographic clustering, DBSCAN and hierarchical comparison results, temporal clustering, PCA metrics, t-SNE evaluation metadata, model artifacts, and registered KMeans models. The notebook uses the outputs produced by the clustering and dimensionality-reduction notebooks so the experiment history remains consistent with the analysis pipeline.


## Step 1 — MLflow Setup

Configure the project tracking database, experiment, reproducibility settings, and paths to the clustering and dimensionality-reduction outputs.


In [1]:
import os
import sys
import json
import tempfile
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

CLUSTERED_DATA_PATH = "../data/processed/chicago_crime_clustered.csv"
REDUCED_DATA_PATH = "../data/processed/chicago_crime_reduced.csv"
PCA_VARIANCE_PATH = "../data/processed/pca_explained_variance.csv"
PCA_IMPORTANCE_PATH = "../data/processed/pca_feature_importance.csv"
PCA_RECONSTRUCTION_PATH = "../data/processed/pca_reconstruction_metrics.csv"
DR_METRICS_PATH = "../data/processed/dimensionality_reduction_metrics.csv"
PCA_METHODOLOGY_PATH = "../data/processed/pca_methodology_summary.csv"

MLFLOW_DB_PATH = os.path.abspath("../mlflow.db")
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.replace(os.sep, '/')}"
EXPERIMENT_NAME = "PatrolIQ_Pipeline_Tracking"

RANDOM_STATE = 42
N_INIT = 10

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print("=" * 70)
print("PATROLIQ — MLFLOW TRACKING SETUP")
print("=" * 70)
print(f"Python version       : {sys.version.split()[0]}")
print(f"MLflow version       : {mlflow.__version__}")
print(f"Scikit-learn version : {sklearn.__version__}")
print(f"Tracking URI         : {mlflow.get_tracking_uri()}")
print(f"Experiment           : {EXPERIMENT_NAME}")
print(f"Experiment ID        : {experiment.experiment_id}")
print("\nSetup completed successfully.")

2026/10/05 21:21:23 INFO mlflow.tracking.fluent: Experiment with name 'PatrolIQ_Pipeline_Tracking' does not exist. Creating a new experiment.


PATROLIQ — MLFLOW TRACKING SETUP
Python version       : 3.13.9
MLflow version       : 3.11.1
Scikit-learn version : 1.7.2
Tracking URI         : sqlite:///E:/PatrolIQ - Smart Safety Analytics Platform/mlflow.db
Experiment           : PatrolIQ_Pipeline_Tracking
Experiment ID        : 3

Setup completed successfully.


## Step 2 — Load and Validate Analysis Outputs

Load the 500,000-record clustered dataset together with the PCA and t-SNE metric artifacts. Validation is performed before any MLflow run is created.


In [2]:
required_files = {
    "Clustered dataset": CLUSTERED_DATA_PATH,
    "Reduced dataset": REDUCED_DATA_PATH,
    "PCA explained variance": PCA_VARIANCE_PATH,
    "PCA feature importance": PCA_IMPORTANCE_PATH,
    "PCA reconstruction metrics": PCA_RECONSTRUCTION_PATH,
    "Dimensionality reduction metrics": DR_METRICS_PATH,
    "PCA methodology": PCA_METHODOLOGY_PATH,
}

print("=" * 70)
print("PATROLIQ — DATA VALIDATION")
print("=" * 70)

missing_files = []
for name, path in required_files.items():
    exists = os.path.exists(path)
    print(f"{'PASS' if exists else 'FAIL'} : {name}")
    if not exists:
        missing_files.append(path)

if missing_files:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing_files))

df_clustered = pd.read_csv(CLUSTERED_DATA_PATH, low_memory=False)
df_reduced = pd.read_csv(REDUCED_DATA_PATH, low_memory=False)
pca_variance_df = pd.read_csv(PCA_VARIANCE_PATH)
pca_importance_df = pd.read_csv(PCA_IMPORTANCE_PATH)
pca_reconstruction_df = pd.read_csv(PCA_RECONSTRUCTION_PATH)
dr_metrics_df = pd.read_csv(DR_METRICS_PATH)
pca_methodology_df = pd.read_csv(PCA_METHODOLOGY_PATH)

assert len(df_clustered) == 500_000
assert len(df_reduced) == 500_000
assert df_clustered["KMeans_Cluster"].nunique() == 9
assert df_clustered["Temporal_Cluster"].nunique() == 4
assert {"PC1", "PC2", "PC3"}.issubset(df_reduced.columns)

pc3 = pca_variance_df.loc[pca_variance_df["Component"] == 3].iloc[0]
pca_variance_3pc = float(pc3["Cumulative_Variance_%"])
assert np.isclose(pca_variance_3pc, 71.772706, atol=0.01)
assert 70 <= pca_variance_3pc <= 80

metric_lookup = dict(zip(dr_metrics_df["Metric"], dr_metrics_df["Value"]))
assert int(metric_lookup["PCA input features"]) == 22
assert int(metric_lookup["PCA components"]) == 3
assert int(metric_lookup["t-SNE records"]) == 20_000

print(f"\nClustered dataset : {df_clustered.shape}")
print(f"Reduced dataset   : {df_reduced.shape}")
print(f"Geographic zones  : {df_clustered['KMeans_Cluster'].nunique()}")
print(f"Temporal patterns : {df_clustered['Temporal_Cluster'].nunique()}")
print(f"PCA dimensions    : {int(metric_lookup['PCA input features'])} -> {int(metric_lookup['PCA components'])}")
print(f"PCA variance      : {pca_variance_3pc:.4f}%")
print(f"t-SNE records     : {int(metric_lookup['t-SNE records']):,}")
print("\nValidation passed.")

PATROLIQ — DATA VALIDATION
PASS : Clustered dataset
PASS : Reduced dataset
PASS : PCA explained variance
PASS : PCA feature importance
PASS : PCA reconstruction metrics
PASS : Dimensionality reduction metrics
PASS : PCA methodology

Clustered dataset : (500000, 41)
Reduced dataset   : (500000, 44)
Geographic zones  : 9
Temporal patterns : 4
PCA dimensions    : 22 -> 3
PCA variance      : 71.7727%
t-SNE records     : 20,000

Validation passed.


## Step 3 — Geographic KMeans Tracking

The geographic patrol-zone model uses five location and administrative features: Latitude, Longitude, Beat, District, and Community Area. The representation is standardized and clustered into nine patrol-oriented geographic zones.

The selected Step 4 evaluation metrics are stored alongside a reproducibility check that compares the rebuilt labels with the saved clustering labels.


In [3]:
GEO_FEATURES = ["Latitude", "Longitude", "Beat", "District", "Community Area"]
GEO_K = 9

X_geo = df_clustered[GEO_FEATURES].astype("float64").copy()
assert X_geo.isna().sum().sum() == 0

geo_scaler = StandardScaler()
X_geo_scaled = geo_scaler.fit_transform(X_geo)

geo_kmeans = KMeans(
    n_clusters=GEO_K,
    random_state=RANDOM_STATE,
    n_init=N_INIT
)
geo_labels = geo_kmeans.fit_predict(X_geo_scaled)

saved_geo_labels = df_clustered["KMeans_Cluster"].astype(int).to_numpy()
geo_ari = adjusted_rand_score(saved_geo_labels, geo_labels)
assert geo_ari >= 0.999

# Metrics recorded during Step 4 model evaluation
GEO_SILHOUETTE = 0.5012
GEO_DBI = 0.7153
GEO_INERTIA = 277202.27

print("=" * 70)
print("GEOGRAPHIC KMEANS — REPRODUCIBILITY CHECK")
print("=" * 70)
print(f"Features               : {GEO_FEATURES}")
print(f"Clusters               : {GEO_K}")
print(f"Training records       : {len(X_geo_scaled):,}")
print(f"Saved-vs-rebuilt ARI   : {geo_ari:.6f}")
print(f"Silhouette score       : {GEO_SILHOUETTE:.4f}")
print(f"Davies-Bouldin index   : {GEO_DBI:.4f}")
print(f"Inertia                : {GEO_INERTIA:,.2f}")

GEOGRAPHIC KMEANS — REPRODUCIBILITY CHECK
Features               : ['Latitude', 'Longitude', 'Beat', 'District', 'Community Area']
Clusters               : 9
Training records       : 500,000
Saved-vs-rebuilt ARI   : 1.000000
Silhouette score       : 0.5012
Davies-Bouldin index   : 0.7153
Inertia                : 277,202.27


## Step 4 — Log Geographic KMeans Run

Store the geographic KMeans parameters, Step 4 evaluation metrics, scaler, cluster profile, and trained model. The model is also registered in MLflow for versioned model management.


In [4]:
with mlflow.start_run(run_name="Geographic_KMeans_K9") as run:
    mlflow.set_tags({
        "project": "PatrolIQ",
        "stage": "Geographic Clustering",
        "algorithm": "KMeans",
        "model_role": "Geographic Patrol Zones",
        "status": "Selected"
    })

    mlflow.log_params({
        "n_clusters": GEO_K,
        "features": ", ".join(GEO_FEATURES),
        "feature_count": len(GEO_FEATURES),
        "scaler": "StandardScaler",
        "random_state": RANDOM_STATE,
        "n_init": N_INIT,
        "training_records": len(X_geo_scaled)
    })

    mlflow.log_metrics({
        "silhouette_score": GEO_SILHOUETTE,
        "davies_bouldin_index": GEO_DBI,
        "inertia": GEO_INERTIA,
        "reproduction_ari": float(geo_ari)
    })

    with tempfile.TemporaryDirectory() as tmpdir:
        scaler_path = os.path.join(tmpdir, "geographic_scaler.joblib")
        profile_path = os.path.join(tmpdir, "geographic_cluster_profile.csv")
        joblib.dump(geo_scaler, scaler_path)

        profile = (
            df_clustered.assign(Tracking_Cluster=geo_labels)
            .groupby("Tracking_Cluster")
            .agg(
                Crime_Count=("Tracking_Cluster", "size"),
                Latitude_Center=("Latitude", "mean"),
                Longitude_Center=("Longitude", "mean"),
                Mean_Severity=("Crime_Severity_Score", "mean"),
                Arrest_Rate=("Arrest", "mean"),
                Domestic_Rate=("Domestic", "mean")
            )
            .reset_index()
        )
        profile.to_csv(profile_path, index=False)
        mlflow.log_artifact(scaler_path, artifact_path="preprocessing")
        mlflow.log_artifact(profile_path, artifact_path="evaluation")

    geo_model_info = mlflow.sklearn.log_model(
        sk_model=geo_kmeans,
        name="geographic_kmeans_model",
        registered_model_name="PatrolIQ_Geographic_KMeans"
    )

    GEO_RUN_ID = run.info.run_id

print("=" * 70)
print("GEOGRAPHIC KMEANS — MLFLOW LOGGING COMPLETE")
print("=" * 70)
print(f"Run ID: {GEO_RUN_ID}")

2026/10/05 21:21:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


GEOGRAPHIC KMEANS — MLFLOW LOGGING COMPLETE
Run ID: ae8495d7e7204b7188b355abedeca188


Successfully registered model 'PatrolIQ_Geographic_KMeans'.
Created version '1' of model 'PatrolIQ_Geographic_KMeans'.


## Step 5 — DBSCAN and Hierarchical Comparison Runs

DBSCAN and hierarchical clustering were evaluated as alternative geographic segmentation approaches. DBSCAN was evaluated on the complete 500,000-record dataset, while hierarchical clustering used a reproducible 10,000-record sample because standard agglomerative clustering is not practical at the full dataset scale.

These runs record the comparison metrics and methodological context used during model selection.


In [5]:
comparison_runs = {}

# DBSCAN — Step 4 evaluation results
with mlflow.start_run(run_name="Geographic_DBSCAN") as run:
    mlflow.set_tags({
        "project": "PatrolIQ",
        "stage": "Geographic Clustering",
        "algorithm": "DBSCAN",
        "model_role": "Comparison",
        "status": "Evaluated"
    })
    mlflow.log_params({
        "features": "X Coordinate, Y Coordinate",
        "scaler": "StandardScaler",
        "evaluation_records": 500000
    })
    mlflow.log_metrics({
        "clusters_excluding_noise": 1049,
        "noise_records": 31847,
        "noise_percentage": 6.37,
        "largest_cluster_percentage": 27.41,
        "silhouette_score": -0.5818
    })
    comparison_runs["Geographic_DBSCAN"] = run.info.run_id

# Hierarchical — Step 4 evaluation results
with mlflow.start_run(run_name="Geographic_Hierarchical_K10") as run:
    mlflow.set_tags({
        "project": "PatrolIQ",
        "stage": "Geographic Clustering",
        "algorithm": "Agglomerative",
        "model_role": "Comparison",
        "status": "Evaluated"
    })
    mlflow.log_params({
        "linkage": "ward",
        "n_clusters": 10,
        "sample_records": 10000,
        "random_state": RANDOM_STATE
    })
    mlflow.log_metrics({
        "silhouette_score": 0.5170,
        "davies_bouldin_index": 0.6240
    })
    comparison_runs["Geographic_Hierarchical_K10"] = run.info.run_id

print("=" * 70)
print("GEOGRAPHIC COMPARISON RUNS — LOGGING COMPLETE")
print("=" * 70)
for name, run_id in comparison_runs.items():
    print(f"{name:<32} : {run_id}")

GEOGRAPHIC COMPARISON RUNS — LOGGING COMPLETE
Geographic_DBSCAN                : 707f24859d2f4811801d8aa8162c7afa
Geographic_Hierarchical_K10      : 682d79346e474a1fbd118269d1951751


## Step 6 — Temporal KMeans Tracking

Temporal KMeans groups records using Hour, Day of Week, and Month. Four temporal patterns are used to summarize recurring crime-time behavior.


In [6]:
TEMPORAL_FEATURES = ["Hour", "Day_of_Week_Num", "Month"]
TEMPORAL_K = 4

X_temporal = df_clustered[TEMPORAL_FEATURES].astype("float64").copy()
assert X_temporal.isna().sum().sum() == 0

temporal_scaler = StandardScaler()
X_temporal_scaled = temporal_scaler.fit_transform(X_temporal)

temporal_kmeans = KMeans(
    n_clusters=TEMPORAL_K,
    random_state=RANDOM_STATE,
    n_init=N_INIT
)
temporal_labels = temporal_kmeans.fit_predict(X_temporal_scaled)

saved_temporal_labels = df_clustered["Temporal_Cluster"].astype(int).to_numpy()
temporal_ari = adjusted_rand_score(saved_temporal_labels, temporal_labels)
assert temporal_ari >= 0.999

TEMPORAL_SILHOUETTE = 0.2702
TEMPORAL_DBI = 1.1319

print("=" * 70)
print("TEMPORAL KMEANS — REPRODUCIBILITY CHECK")
print("=" * 70)
print(f"Features               : {TEMPORAL_FEATURES}")
print(f"Clusters               : {TEMPORAL_K}")
print(f"Saved-vs-rebuilt ARI   : {temporal_ari:.6f}")
print(f"Silhouette score       : {TEMPORAL_SILHOUETTE:.4f}")
print(f"Davies-Bouldin index   : {TEMPORAL_DBI:.4f}")

TEMPORAL KMEANS — REPRODUCIBILITY CHECK
Features               : ['Hour', 'Day_of_Week_Num', 'Month']
Clusters               : 4
Saved-vs-rebuilt ARI   : 1.000000
Silhouette score       : 0.2702
Davies-Bouldin index   : 1.1319


## Step 7 — Log Temporal KMeans Run

Store the temporal model parameters, evaluation metrics, preprocessing scaler, and trained KMeans model. The model is registered separately from the geographic model.


In [7]:
with mlflow.start_run(run_name="Temporal_KMeans_K4") as run:
    mlflow.set_tags({
        "project": "PatrolIQ",
        "stage": "Temporal Clustering",
        "algorithm": "KMeans",
        "model_role": "Temporal Crime Patterns",
        "status": "Selected"
    })

    mlflow.log_params({
        "n_clusters": TEMPORAL_K,
        "features": ", ".join(TEMPORAL_FEATURES),
        "feature_count": len(TEMPORAL_FEATURES),
        "scaler": "StandardScaler",
        "random_state": RANDOM_STATE,
        "n_init": N_INIT,
        "training_records": len(X_temporal_scaled)
    })

    mlflow.log_metrics({
        "silhouette_score": TEMPORAL_SILHOUETTE,
        "davies_bouldin_index": TEMPORAL_DBI,
        "inertia": float(temporal_kmeans.inertia_),
        "reproduction_ari": float(temporal_ari)
    })

    with tempfile.TemporaryDirectory() as tmpdir:
        scaler_path = os.path.join(tmpdir, "temporal_scaler.joblib")
        joblib.dump(temporal_scaler, scaler_path)
        mlflow.log_artifact(scaler_path, artifact_path="preprocessing")

    temporal_model_info = mlflow.sklearn.log_model(
        sk_model=temporal_kmeans,
        name="temporal_kmeans_model",
        registered_model_name="PatrolIQ_Temporal_KMeans"
    )

    TEMPORAL_RUN_ID = run.info.run_id

print("=" * 70)
print("TEMPORAL KMEANS — MLFLOW LOGGING COMPLETE")
print("=" * 70)
print(f"Run ID: {TEMPORAL_RUN_ID}")

2026/10/05 21:22:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


TEMPORAL KMEANS — MLFLOW LOGGING COMPLETE
Run ID: c9a79842f77e48a091909b121860c534


Successfully registered model 'PatrolIQ_Temporal_KMeans'.
Created version '1' of model 'PatrolIQ_Temporal_KMeans'.


## Step 8 — PCA and t-SNE Tracking

The dimensionality-reduction run records the 22-dimensional PCA input, three-component variance retention, reconstruction error, feature-importance artifact, and t-SNE evaluation metadata.

PCA uses `RobustScaler` with the 25th–75th percentile range. The three retained principal components explain 71.77% of total variance. t-SNE is tracked as a visualization/evaluation technique rather than a deployment model.


In [8]:
PCA_FEATURES = [
    "Latitude", "Longitude", "X Coordinate", "Y Coordinate",
    "Beat", "District", "Ward", "Community Area",
    "Latitude_Bin", "Longitude_Bin", "Month", "Year",
    "Date_DayOfYear", "Date_Quarter", "Month_Sin", "Month_Cos",
    "Arrest", "Domestic", "Primary_Type_Frequency",
    "IUCR_Frequency", "FBI_Code_Frequency", "DayOfYear_Normalized"
]

PCA_COMPONENTS = 3
PCA_VARIANCE = float(metric_lookup["PCA 3-PC variance %"])
PCA_MSE = float(metric_lookup["PCA reconstruction MSE"])
PCA_RMSE = float(metric_lookup["PCA reconstruction RMSE"])
PCA_MAE = float(metric_lookup["PCA reconstruction MAE"])
TSNE_RECORDS = int(metric_lookup["t-SNE records"])
TSNE_KL = float(metric_lookup["t-SNE KL divergence"])

assert len(PCA_FEATURES) == 22
assert 70 <= PCA_VARIANCE <= 80

with mlflow.start_run(run_name="Dimensionality_Reduction_PCA_TSNE") as run:
    mlflow.set_tags({
        "project": "PatrolIQ",
        "stage": "Dimensionality Reduction",
        "primary_technique": "PCA",
        "secondary_technique": "t-SNE",
        "status": "Selected"
    })

    mlflow.log_params({
        "pca_input_features": len(PCA_FEATURES),
        "pca_components": PCA_COMPONENTS,
        "pca_scaler": "RobustScaler",
        "pca_quantile_range": "25,75",
        "pca_features": ", ".join(PCA_FEATURES),
        "tsne_records": TSNE_RECORDS
    })

    mlflow.log_metrics({
        "pca_pc1_variance_pct": float(pca_variance_df.loc[pca_variance_df['Component'] == 1, 'Explained_Variance_%'].iloc[0]),
        "pca_pc2_variance_pct": float(pca_variance_df.loc[pca_variance_df['Component'] == 2, 'Explained_Variance_%'].iloc[0]),
        "pca_pc3_variance_pct": float(pca_variance_df.loc[pca_variance_df['Component'] == 3, 'Explained_Variance_%'].iloc[0]),
        "pca_3pc_cumulative_variance_pct": PCA_VARIANCE,
        "pca_reconstruction_mse": PCA_MSE,
        "pca_reconstruction_rmse": PCA_RMSE,
        "pca_reconstruction_mae": PCA_MAE,
        "tsne_kl_divergence": TSNE_KL
    })

    mlflow.log_artifact(PCA_VARIANCE_PATH, artifact_path="pca")
    mlflow.log_artifact(PCA_IMPORTANCE_PATH, artifact_path="pca")
    mlflow.log_artifact(PCA_RECONSTRUCTION_PATH, artifact_path="pca")
    mlflow.log_artifact(DR_METRICS_PATH, artifact_path="metrics")
    mlflow.log_artifact(PCA_METHODOLOGY_PATH, artifact_path="methodology")

    DR_RUN_ID = run.info.run_id

print("=" * 70)
print("DIMENSIONALITY REDUCTION — MLFLOW LOGGING COMPLETE")
print("=" * 70)
print(f"PCA input dimensions : {len(PCA_FEATURES)}")
print(f"PCA components       : {PCA_COMPONENTS}")
print(f"PCA variance         : {PCA_VARIANCE:.4f}%")
print(f"Reconstruction RMSE  : {PCA_RMSE:.6f}")
print(f"t-SNE records        : {TSNE_RECORDS:,}")
print(f"t-SNE KL divergence  : {TSNE_KL:.6f}")
print(f"Run ID               : {DR_RUN_ID}")

DIMENSIONALITY REDUCTION — MLFLOW LOGGING COMPLETE
PCA input dimensions : 22
PCA components       : 3
PCA variance         : 71.7727%
Reconstruction RMSE  : 0.345787
t-SNE records        : 20,000
t-SNE KL divergence  : 1.075575
Run ID               : a69a61db856c41dfbdc971a6883e6e62


## Step 9 — Experiment Comparison

Retrieve the experiment runs and display the parameters and metrics needed to compare the tracked PatrolIQ methods.


In [9]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time ASC"]
)

print("=" * 70)
print("MLFLOW EXPERIMENT — RUN COMPARISON")
print("=" * 70)
print(f"Experiment : {EXPERIMENT_NAME}")
print(f"Runs found : {len(runs)}")

display_columns = [
    "tags.mlflow.runName",
    "status",
    "tags.algorithm",
    "tags.stage",
    "metrics.silhouette_score",
    "metrics.davies_bouldin_index",
    "metrics.pca_3pc_cumulative_variance_pct",
    "metrics.pca_reconstruction_rmse",
    "metrics.tsne_kl_divergence"
]

available_columns = [c for c in display_columns if c in runs.columns]
print("\n" + runs[available_columns].to_string(index=False))

MLFLOW EXPERIMENT — RUN COMPARISON
Experiment : PatrolIQ_Pipeline_Tracking
Runs found : 5

              tags.mlflow.runName   status tags.algorithm               tags.stage  metrics.silhouette_score  metrics.davies_bouldin_index  metrics.pca_3pc_cumulative_variance_pct  metrics.pca_reconstruction_rmse  metrics.tsne_kl_divergence
             Geographic_KMeans_K9 FINISHED         KMeans    Geographic Clustering                    0.5012                        0.7153                                      NaN                              NaN                         NaN
                Geographic_DBSCAN FINISHED         DBSCAN    Geographic Clustering                   -0.5818                           NaN                                      NaN                              NaN                         NaN
      Geographic_Hierarchical_K10 FINISHED  Agglomerative    Geographic Clustering                    0.5170                        0.6240                                      NaN       

## Step 10 — Validation Summary

Confirm that the expected clustering and dimensionality-reduction runs completed successfully and that the two selected KMeans models are available in the MLflow model registry.


In [10]:
expected_runs = {
    "Geographic_KMeans_K9",
    "Geographic_DBSCAN",
    "Geographic_Hierarchical_K10",
    "Temporal_KMeans_K4",
    "Dimensionality_Reduction_PCA_TSNE"
}

runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
actual_runs = set(runs["tags.mlflow.runName"].dropna())
missing_runs = expected_runs - actual_runs
assert not missing_runs, f"Missing MLflow runs: {missing_runs}"

tracked = runs[runs["tags.mlflow.runName"].isin(expected_runs)]
assert (tracked["status"] == "FINISHED").all(), "One or more tracked runs did not finish successfully."

client = mlflow.MlflowClient()
registered_names = {rm.name for rm in client.search_registered_models()}
expected_models = {"PatrolIQ_Geographic_KMeans", "PatrolIQ_Temporal_KMeans"}
missing_models = expected_models - registered_names
assert not missing_models, f"Missing registered models: {missing_models}"

checks = {
    "Geographic KMeans K=9 run": "Geographic_KMeans_K9" in actual_runs,
    "DBSCAN comparison run": "Geographic_DBSCAN" in actual_runs,
    "Hierarchical comparison run": "Geographic_Hierarchical_K10" in actual_runs,
    "Temporal KMeans K=4 run": "Temporal_KMeans_K4" in actual_runs,
    "PCA + t-SNE run": "Dimensionality_Reduction_PCA_TSNE" in actual_runs,
    "Geographic model registered": "PatrolIQ_Geographic_KMeans" in registered_names,
    "Temporal model registered": "PatrolIQ_Temporal_KMeans" in registered_names,
    "PCA 3-PC variance 70–80%": 70 <= PCA_VARIANCE <= 80,
    "PCA reconstruction metrics logged": np.isfinite(PCA_RMSE),
    "t-SNE metric logged": np.isfinite(TSNE_KL)
}

print("=" * 70)
print("PATROLIQ — MLFLOW VALIDATION SUMMARY")
print("=" * 70)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} : {name}")

passed_count = sum(checks.values())
print(f"\nChecks passed: {passed_count}/{len(checks)}")

assert all(checks.values()), "One or more MLflow validation checks failed."
print("\nMLflow experiment tracking completed successfully.")

PATROLIQ — MLFLOW VALIDATION SUMMARY
PASS : Geographic KMeans K=9 run
PASS : DBSCAN comparison run
PASS : Hierarchical comparison run
PASS : Temporal KMeans K=4 run
PASS : PCA + t-SNE run
PASS : Geographic model registered
PASS : Temporal model registered
PASS : PCA 3-PC variance 70–80%
PASS : PCA reconstruction metrics logged
PASS : t-SNE metric logged

Checks passed: 10/10

MLflow experiment tracking completed successfully.
